<a href="https://colab.research.google.com/github/Ivan8Garcia/Proyectos_MachineLearning/blob/main/Curso_Agentes_de_IA_y_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Conexion con LLM´s**

In [7]:
!pip install -q langchain langchain-google-genai google-generativeai

In [8]:
from google.colab import userdata
GEMINI_API_KEY=userdata.get('Gemini_API_key')

In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm= ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=0,
    google_api_key=GEMINI_API_KEY
)

In [ ]:
respuesta= llm.invoke("que es el RAG en IA?")
respuesta

AIMessage(content=[{'type': 'text', 'text': '**RAG** son las siglas en inglés de **Retrieval-Augmented Generation** (en español: **Generación Aumentada por Recuperación**).\n\nEs una técnica en Inteligencia Artificial diseñada para que los modelos de lenguaje (como ChatGPT o Claude) sean más precisos, actuales y confiables al conectarlos con fuentes de datos externas que no formaron parte de su entrenamiento original.\n\nAquí te explico cómo funciona, por qué es importante y un ejemplo sencillo:\n\n---\n\n### 1. ¿Por qué es necesario el RAG?\nLos modelos de lenguaje (LLM) tienen dos limitaciones principales:\n*   **Fecha de corte:** Solo saben lo que aprendieron hasta el día en que terminó su entrenamiento (por ejemplo, si un modelo terminó de entrenarse en 2023, no sabe qué pasó ayer).\n*   **Alucinaciones:** A veces inventan datos con mucha seguridad si no conocen la respuesta.\n*   **Falta de datos privados:** No conocen los documentos internos de tu empresa o tus archivos personale

In [ ]:
PROMPT_TRIAJE="""

"""



In [ ]:
from typing import Literal,List,Dict
from pydantic import BaseModel,Field


class TriajeOut(BaseModel):
  decision: Literal["AUTO_RESOLVER", "PEDIR_INFO","ABRIR_TICKET"]
  urgencia: Literal["BAJA","MEDIANA","ALTA"]
  campos_faltantes:List[str]=Field(default_factory=list)

In [ ]:
from langchain_core.messages import SystemMessage,HumanMessage

chain_de_triaje= llm.with_structured_output(TriajeOut)

def triaje(mensaje:str) -> Dict:
  salida: TriajeOut= chain_de_triaje.invoke(
      [
          SystemMessage(content=PROMPT_TRIAJE),
          HumanMessage(content=mensaje)
      ]
  )
  return salida.model_dump()

In [ ]:
mensajes_de_prueba=[
    "Puedo obterner un reembolso por el internet de mi home office?",
    "Quiero una excepcion  para teletrabajar durante 5 dias",
    "Como funciona la politica de comidas para viajes?",
    "Existe una politica para anticipos de vacaciones?",
    "Quien fue napoleon bonaparte?"
]

for pregunta in mensajes_de_prueba:
  r=triaje(pregunta)
  print(f"{pregunta} -> {r}")

#**RAG**

In [1]:
!pip install -q langchain_community faiss-cpu langchain-text-splitters pymupdf

In [3]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

docs=[]

for documento in Path("/content/").glob("*.pdf"):
    try:
        loader = PyMuPDFLoader(str(documento))
        docs.extend(loader.load())
        print(f"Archivo cargado: {documento.name}")
    except Exception as e:
        print(f"Error cargando archivo: {documento.name}: {e}")

print(f"Total de documentos cargados: {len(docs)}")


Archivo cargado: Política de Reembolsos (Viajes y Gastos).pdf
Archivo cargado: Política de Teletrabajo (Home Office).pdf
Archivo cargado: Política de Uso de Correo Electrónico y Seguridad de la Información.pdf
Total de documentos cargados: 3


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter= RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=30)
chunks = splitter.split_documents(docs)

In [5]:
for chunk in chunks:
  print(chunk)
  print("-------------------")

page_content='Política de Reembolsos (Viajes y 
Gastos) 
1. Objetivo Establecer las directrices y procedimientos para el reembolso de gastos 
incurridos por los empleados en el ejercicio de sus funciones oficiales, asegurando la 
transparencia, equidad y cumplimiento fiscal.' metadata={'producer': 'Skia/PDF m143 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '/content/Política de Reembolsos (Viajes y Gastos).pdf', 'file_path': '/content/Política de Reembolsos (Viajes y Gastos).pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'Política de Reembolsos (Viajes y Gastos)', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}
-------------------
page_content='2. Ámbito de Aplicación Esta política se aplica a todos los empleados fijos y temporales 
que incurran en gastos en nombre de la empresa. 
3. Gastos de Viaje 
●​ Transporte Aéreo: Se reembolsarán vuelos en clase económica. Cualquier mejora' met

In [11]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

modelo_embeddings= GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GEMINI_API_KEY
)

In [12]:
from langchain_community.vectorstores import FAISS

vectorstore= FAISS.from_documents(chunks,modelo_embeddings)

retriever= vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold":0.3, "k":4}
)

In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

prompt_rag = ChatPromptTemplate(
    [
        (
            "system",
            """
            Eres el especialista en RR.HH. de la empresa Carraro Desarrollo de Software.
            Responde siempre utilizando los conocimientos de las bases de datos pasadas a ti.
            Si no hay información sobre la pregunta en los datos, responde sólo 'No lo sé'.
            """
        ),
        ("human", "Contexto: {context}.\nPregunta del empleado: {input}")
    ]
)

document_chain = create_stuff_documents_chain(llm,prompt_rag)

ModuleNotFoundError: No module named 'langchain.chains'